# Части 3-4: CRNN для распознавания номерных знаков

**Часть 3** — подготовка датасета кропов.  
**Часть 4** — обучение CRNN (baseline + 3 эксперимента + финальная модель).

In [1]:
%matplotlib inline

import os
import math
import random
import json
import glob
from pathlib import Path

import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

DATASET_ROOT = r"C:\ITMO\autoriaNumberplateOcrRu\autoriaNumberplateOcrRu"

BATCH_SIZE   = 32
IMG_HEIGHT   = 32
IMG_WIDTH    = 128
NUM_WORKERS  = 0
RANDOM_SEED  = 42

MAX_TRAIN_SAMPLES = 3000
MAX_TEST_SAMPLES  = 300

CHECKPOINT_DIR = Path(r"C:\ITMO\designing_neural_network_architectures_2025_02\seminar_01\crnn_checkpoints")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

Device: cuda


## Часть 3: Датасет (autoriaNumberplateOcrRu)

Датасет содержит кропы номеров в формате `split/{img,ann}/`, где каждый `.json`
имеет поле `description` (текст номера) и `name` (имя изображения без расширения).

In [2]:
ALPHABET = "0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ"
blank_idx  = 0
char2idx   = {c: i + 1 for i, c in enumerate(ALPHABET)}
idx2char   = {i + 1: c for i, c in enumerate(ALPHABET)}
vocab_size = len(ALPHABET) + 1

train_transforms = T.Compose([
    T.Grayscale(num_output_channels=1),
    T.Resize((IMG_HEIGHT, IMG_WIDTH)),
    T.ToTensor(),
    T.Normalize((0.5,), (0.5,))
])
test_transforms = train_transforms

class NumberplateOCRDataset(Dataset):
    def __init__(self, root, split, transforms=None, max_samples=None):
        self.root = root
        self.split = split
        self.transforms = transforms
        ann_dir = os.path.join(root, split, "ann")
        self.ann_paths = glob.glob(os.path.join(ann_dir, "*.json"))
        self.ann_paths.sort()
        if max_samples is not None:
            self.ann_paths = self.ann_paths[:max_samples]

    def __len__(self):
        return len(self.ann_paths)

    def encode_text(self, text):
        return [char2idx[c] for c in str(text).upper() if c in char2idx]

    def __getitem__(self, idx):
        ann_path = self.ann_paths[idx]
        with open(ann_path, "r", encoding="utf-8") as f:
            data = json.load(f)
        name = data["name"]
        text = str(data.get("description", "")).upper()
        img_path = os.path.join(self.root, self.split, "img", name + ".png")
        image = Image.open(img_path).convert("RGB")
        if self.transforms is not None:
            image = self.transforms(image)
        label = torch.tensor(self.encode_text(text), dtype=torch.long)
        return image, label, text

def collate_fn(batch):
    batch = [b for b in batch if len(b[1]) > 0]
    images  = torch.stack([b[0] for b in batch], dim=0)
    labels  = [b[1] for b in batch]
    texts   = [b[2] for b in batch]
    label_lengths = torch.tensor([len(l) for l in labels], dtype=torch.long)
    targets = torch.cat(labels, dim=0)
    return images, targets, label_lengths, texts

train_dataset = NumberplateOCRDataset(DATASET_ROOT, "train", train_transforms, MAX_TRAIN_SAMPLES)
test_dataset  = NumberplateOCRDataset(DATASET_ROOT, "test",  test_transforms,  MAX_TEST_SAMPLES)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, collate_fn=collate_fn, drop_last=False)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, collate_fn=collate_fn, drop_last=False)

print(f"Train: {len(train_dataset)}, Test: {len(test_dataset)}")
print(f"Vocab size: {vocab_size}, ALPHABET: {ALPHABET}")

# Проверка одного примера
img, lbl, txt = train_dataset[0]
print(f"Sample: '{txt}', label shape: {lbl.shape}, image shape: {img.shape}")

Train: 3000, Test: 300
Vocab size: 37, ALPHABET: 0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ
Sample: 'A088KK60', label shape: torch.Size([8]), image shape: torch.Size([1, 32, 128])


## Часть 4.1: Модель CRNN (baseline)

In [3]:
class CRNN(nn.Module):
    """
    CNN backbone → BiLSTM → CTC.
    Input:  (B, 1, H, W), default H=32, W=128.
    Output: (T, B, vocab_size) log-probabilities.
    """
    def __init__(self, vocab_size, img_height=32, rnn_hidden=256, rnn_layers=2, cnn_channels=(64, 128, 256, 256)):
        super().__init__()
        self.cnn = self._build_cnn(img_height, cnn_channels)
        # После CNN высота схлопывается до 1, число каналов = cnn_channels[-1]
        cnn_out_channels = cnn_channels[-1]
        self.rnn = nn.LSTM(
            input_size=cnn_out_channels,
            hidden_size=rnn_hidden,
            num_layers=rnn_layers,
            batch_first=False,
            bidirectional=True,
        )
        self.fc = nn.Linear(rnn_hidden * 2, vocab_size)

    def _build_cnn(self, img_height, chs):
        # 4 блока Conv+BN+ReLU, MaxPool только по высоте первые 3 раза
        # Итог для H=32: 32 → 16 → 8 → 4 → 1 (adaptive pool)
        layers = []
        in_ch = 1
        pool_strides = [(2, 2), (2, 2), (2, 1), None]  # None → adaptive
        for i, out_ch in enumerate(chs):
            layers += [
                nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
                nn.BatchNorm2d(out_ch),
                nn.ReLU(inplace=True),
            ]
            if pool_strides[i] is not None:
                layers.append(nn.MaxPool2d(kernel_size=pool_strides[i], stride=pool_strides[i]))
            in_ch = out_ch
        # Финальный адаптивный пулинг до высоты=1
        layers.append(nn.AdaptiveAvgPool2d((1, None)))
        return nn.Sequential(*layers)

    def forward(self, x):
        feat = self.cnn(x)                   # (B, C, 1, W')
        feat = feat.squeeze(2)               # (B, C, W')
        feat = feat.permute(2, 0, 1)         # (W', B, C)  = (T, B, C)
        out, _ = self.rnn(feat)              # (T, B, 2*H)
        logits = self.fc(out)                # (T, B, vocab)
        return logits.log_softmax(2)         # log-probs for CTC

In [4]:
def compute_cer(predictions, ground_truths):
    """Character Error Rate (Levenshtein / len(gt))."""
    def levenshtein(a, b):
        m, n = len(a), len(b)
        dp = list(range(n + 1))
        for i in range(1, m + 1):
            prev, dp[0] = dp[0], i
            for j in range(1, n + 1):
                prev, dp[j] = dp[j], min(
                    dp[j] + 1,
                    dp[j - 1] + 1,
                    prev + (0 if a[i - 1] == b[j - 1] else 1)
                )
        return dp[n]

    total_dist, total_len = 0, 0
    for pred, gt in zip(predictions, ground_truths):
        total_dist += levenshtein(pred, gt)
        total_len  += max(len(gt), 1)
    return total_dist / total_len

def decode_sequence(seq):
    seq = seq.cpu().numpy().tolist()
    prev, out = None, []
    for s in seq:
        if s != blank_idx and s != prev and s in idx2char:
            out.append(idx2char[s])
        prev = s
    return "".join(out)

In [7]:
def train_epoch(model, loader, optimizer, criterion, scheduler=None):
    model.train()
    running_loss = 0.0
    for images, targets, target_lengths, _ in loader:
        images = images.to(device)
        targets = targets.to(device)
        target_lengths = target_lengths.to(device)
        logits = model(images)
        T_cur, N_cur, _ = logits.size()
        input_lengths = torch.full((N_cur,), T_cur, dtype=torch.long, device=device)
        loss = criterion(logits, targets, input_lengths, target_lengths)
        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()
        running_loss += loss.item()
    if scheduler is not None:
        scheduler.step()
    return running_loss / max(1, len(loader))

def evaluate_cer(model, loader, max_batches=20):
    model.eval()
    preds, gts = [], []
    with torch.no_grad():
        for bi, (images, _, _, texts) in enumerate(loader):
            images = images.to(device)
            logits = model(images)
            pred_seq = logits.permute(1, 0, 2).argmax(2)
            for i in range(images.size(0)):
                preds.append(decode_sequence(pred_seq[i]))
                gts.append(texts[i])
            if bi + 1 >= max_batches:
                break
    cer = compute_cer(preds, gts)
    exact = sum(p == g for p, g in zip(preds, gts)) / max(1, len(preds))
    return cer, exact, preds, gts

def run_experiment(name, model, n_epochs, lr=1e-3, scheduler_fn=None,
                   train_loader=train_loader, test_loader=test_loader,
                   print_every=5):
    criterion = nn.CTCLoss(blank=blank_idx, zero_infinity=True)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    scheduler = scheduler_fn(optimizer) if scheduler_fn else None

    print(f"Experiment: {name}")

    best_cer = float("inf")
    for epoch in range(1, n_epochs + 1):
        loss = train_epoch(model, train_loader, optimizer, criterion, scheduler)
        if epoch % print_every == 0 or epoch == n_epochs:
            cer, exact, _, _ = evaluate_cer(model, test_loader)
            print(f"Epoch {epoch:3d}/{n_epochs} | loss: {loss:.4f} | CER: {cer:.4f} | Exact: {exact*100:.1f}%")
            if cer < best_cer:
                best_cer = cer

    cer, exact, preds, gts = evaluate_cer(model, test_loader, max_batches=99)
    print(f"\nFinal CER: {cer:.4f} | Exact accuracy: {exact*100:.2f}%")
    print("Sample predictions:")
    for gt, pred in zip(gts[:10], preds[:10]):
        mark = "✓" if gt == pred else "✗"
        print(f"  {mark} GT: {gt:12s} | PRED: {pred}")
    return cer, exact

## Baseline (20 эпох)

In [8]:
baseline_model = CRNN(vocab_size=vocab_size).to(device)
n_params = sum(p.numel() for p in baseline_model.parameters())
print(f"CRNN parameters: {n_params:,}")

baseline_cer, baseline_exact = run_experiment(
    name="Baseline (128x32, Adam lr=1e-3, 20 epochs)",
    model=baseline_model,
    n_epochs=20,
    lr=1e-3,
)

CRNN parameters: 3,609,061
Experiment: Baseline (128x32, Adam lr=1e-3, 20 epochs)
Epoch   5/20 | loss: 0.6232 | CER: 0.3925 | Exact: 0.0%
Epoch  10/20 | loss: 0.0476 | CER: 0.1152 | Exact: 43.0%
Epoch  15/20 | loss: 0.0089 | CER: 0.1236 | Exact: 46.0%
Epoch  20/20 | loss: 0.0127 | CER: 0.0642 | Exact: 62.3%

Final CER: 0.0642 | Exact accuracy: 62.33%
Sample predictions:
  ✓ GT: A001BP54     | PRED: A001BP54
  ✓ GT: A001PC71     | PRED: A001PC71
  ✗ GT: A002KX152    | PRED: A002KX12
  ✗ GT: A002XC763    | PRED: A002XCO35
  ✗ GT: A002XY89     | PRED: A002X9
  ✗ GT: A003CX196    | PRED: A003CX35
  ✓ GT: A004OE23     | PRED: A004OE23
  ✓ GT: A005AX26     | PRED: A005AX26
  ✓ GT: A006AA10     | PRED: A006AA10
  ✗ GT: A007AE799    | PRED: A007AE199


## Часть 4.2: Эксперименты

### Эксперимент 1 — Увеличение ширины входа (IMG_WIDTH=160)

In [9]:
W1 = 160
transforms_w160 = T.Compose([
    T.Grayscale(num_output_channels=1),
    T.Resize((IMG_HEIGHT, W1)),
    T.ToTensor(),
    T.Normalize((0.5,), (0.5,))
])

train_ds_w160  = NumberplateOCRDataset(DATASET_ROOT, "train", transforms_w160, MAX_TRAIN_SAMPLES)
test_ds_w160   = NumberplateOCRDataset(DATASET_ROOT, "test",  transforms_w160, MAX_TEST_SAMPLES)
train_ldr_w160 = DataLoader(train_ds_w160, batch_size=BATCH_SIZE, shuffle=True,
                             num_workers=NUM_WORKERS, collate_fn=collate_fn, drop_last=False)
test_ldr_w160  = DataLoader(test_ds_w160,  batch_size=BATCH_SIZE, shuffle=False,
                             num_workers=NUM_WORKERS, collate_fn=collate_fn, drop_last=False)

exp1_model = CRNN(vocab_size=vocab_size).to(device)
exp1_cer, exp1_exact = run_experiment(
    name="Exp1: IMG_WIDTH=160",
    model=exp1_model,
    n_epochs=20,
    lr=1e-3,
    train_loader=train_ldr_w160,
    test_loader=test_ldr_w160,
)

Experiment: Exp1: IMG_WIDTH=160
Epoch   5/20 | loss: 0.9605 | CER: 0.4257 | Exact: 0.0%
Epoch  10/20 | loss: 0.1331 | CER: 0.2598 | Exact: 7.7%
Epoch  15/20 | loss: 0.0254 | CER: 0.1929 | Exact: 15.7%
Epoch  20/20 | loss: 0.0093 | CER: 0.1636 | Exact: 21.7%

Final CER: 0.1636 | Exact accuracy: 21.67%
Sample predictions:
  ✓ GT: A001BP54     | PRED: A001BP54
  ✗ GT: A001PC71     | PRED: A001PC77
  ✗ GT: A002KX152    | PRED: A002KXX152
  ✗ GT: A002XC763    | PRED: A002XO21
  ✗ GT: A002XY89     | PRED: A002X139
  ✗ GT: A003CX196    | PRED: A003CX35
  ✓ GT: A004OE23     | PRED: A004OE23
  ✗ GT: A005AX26     | PRED: A005AAX36
  ✗ GT: A006AA10     | PRED: A006AA00
  ✗ GT: A007AE799    | PRED: A007AE199


### Эксперимент 2 — CosineAnnealingLR scheduler

In [10]:
exp2_model = CRNN(vocab_size=vocab_size).to(device)
exp2_cer, exp2_exact = run_experiment(
    name="Exp2: CosineAnnealingLR (T_max=20)",
    model=exp2_model,
    n_epochs=20,
    lr=1e-3,
    scheduler_fn=lambda opt: optim.lr_scheduler.CosineAnnealingLR(opt, T_max=20, eta_min=1e-5),
)

Experiment: Exp2: CosineAnnealingLR (T_max=20)
Epoch   5/20 | loss: 0.8152 | CER: 0.3964 | Exact: 0.0%
Epoch  10/20 | loss: 0.0713 | CER: 0.2127 | Exact: 14.3%
Epoch  15/20 | loss: 0.0143 | CER: 0.1782 | Exact: 17.0%
Epoch  20/20 | loss: 0.0098 | CER: 0.1754 | Exact: 19.0%

Final CER: 0.1754 | Exact accuracy: 19.00%
Sample predictions:
  ✓ GT: A001BP54     | PRED: A001BP54
  ✗ GT: A001PC71     | PRED: A001PC77
  ✗ GT: A002KX152    | PRED: A002KX154
  ✗ GT: A002XC763    | PRED: A002C75
  ✗ GT: A002XY89     | PRED: A002X199
  ✗ GT: A003CX196    | PRED: A003C35
  ✗ GT: A004OE23     | PRED: A004OOE22
  ✗ GT: A005AX26     | PRED: A005AX36
  ✗ GT: A006AA10     | PRED: A006AA100
  ✗ GT: A007AE799    | PRED: A007AE199


### Эксперимент 3 — Аугментации (RandomAffine + ColorJitter + GaussianBlur)

In [11]:
train_transforms_aug = T.Compose([
    T.Grayscale(num_output_channels=1),
    T.Resize((IMG_HEIGHT, IMG_WIDTH)),
    T.RandomAffine(degrees=5, translate=(0.05, 0.05), shear=5),
    T.ColorJitter(brightness=0.3, contrast=0.3),
    T.GaussianBlur(kernel_size=3, sigma=(0.1, 1.0)),
    T.ToTensor(),
    T.Normalize((0.5,), (0.5,))
])

train_ds_aug  = NumberplateOCRDataset(DATASET_ROOT, "train", train_transforms_aug, MAX_TRAIN_SAMPLES)
train_ldr_aug = DataLoader(train_ds_aug, batch_size=BATCH_SIZE, shuffle=True,
                            num_workers=NUM_WORKERS, collate_fn=collate_fn, drop_last=False)

exp3_model = CRNN(vocab_size=vocab_size).to(device)
exp3_cer, exp3_exact = run_experiment(
    name="Exp3: Augmentations (Affine + ColorJitter + GaussianBlur)",
    model=exp3_model,
    n_epochs=20,
    lr=1e-3,
    train_loader=train_ldr_aug,
)

Experiment: Exp3: Augmentations (Affine + ColorJitter + GaussianBlur)
Epoch   5/20 | loss: 0.9018 | CER: 0.4305 | Exact: 0.0%
Epoch  10/20 | loss: 0.1050 | CER: 0.2246 | Exact: 15.0%
Epoch  15/20 | loss: 0.0414 | CER: 0.1311 | Exact: 31.7%
Epoch  20/20 | loss: 0.0229 | CER: 0.1204 | Exact: 33.3%

Final CER: 0.1204 | Exact accuracy: 33.33%
Sample predictions:
  ✓ GT: A001BP54     | PRED: A001BP54
  ✗ GT: A001PC71     | PRED: A001PC77
  ✓ GT: A002KX152    | PRED: A002KX152
  ✗ GT: A002XC763    | PRED: A002C76
  ✗ GT: A002XY89     | PRED: A002XX59
  ✗ GT: A003CX196    | PRED: A003CX78
  ✗ GT: A004OE23     | PRED: A004OE22
  ✗ GT: A005AX26     | PRED: A005AX2
  ✗ GT: A006AA10     | PRED: A006AA100
  ✗ GT: A007AE799    | PRED: A007AE199


### Сводная таблица экспериментов

In [12]:
results = [
    ("Baseline",          "128×32, Adam lr=1e-3",                                baseline_cer, baseline_exact),
    ("Exp1: Width 160",   "IMG_WIDTH=160",                                        exp1_cer,     exp1_exact),
    ("Exp2: Cosine LR",   "CosineAnnealingLR(T_max=20)",                         exp2_cer,     exp2_exact),
    ("Exp3: Augment",     "RandomAffine+ColorJitter+GaussianBlur",               exp3_cer,     exp3_exact),
]

print(f"{'Experiment':<20} {'Config':<40} {'CER':>6} {'Exact%':>8}")
for name, cfg, cer, exact in results:
    print(f"{name:<20} {cfg:<40} {cer:>6.4f} {exact*100:>7.2f}%")

Experiment           Config                                      CER   Exact%
Baseline             128×32, Adam lr=1e-3                     0.0642   62.33%
Exp1: Width 160      IMG_WIDTH=160                            0.1636   21.67%
Exp2: Cosine LR      CosineAnnealingLR(T_max=20)              0.1754   19.00%
Exp3: Augment        RandomAffine+ColorJitter+GaussianBlur    0.1204   33.33%


## Часть 4.3: Финальная модель (лучшая конфигурация, 50 эпох)

In [17]:
best_idx = int(np.argmin([r[2] for r in results]))
best_name = results[best_idx][0]
print(f"Best experiment: {best_name} (CER={results[best_idx][2]:.4f})")

final_model = CRNN(vocab_size=vocab_size).to(device)

final_train_loader = train_loader
final_test_loader  = test_loader

final_cer, final_exact = run_experiment(
    name="Final: Baseline",
    model=final_model,
    n_epochs=50,
    lr=1e-3,
    train_loader=final_train_loader,
    test_loader=final_test_loader,
    print_every=5,
)

Best experiment: Baseline (CER=0.0642)
Experiment: Final: Baseline
Epoch   5/50 | loss: 0.5365 | CER: 0.3687 | Exact: 0.7%
Epoch  10/50 | loss: 0.0346 | CER: 0.1893 | Exact: 20.0%
Epoch  15/50 | loss: 0.0142 | CER: 0.1442 | Exact: 32.3%
Epoch  20/50 | loss: 0.0278 | CER: 0.1485 | Exact: 30.3%
Epoch  25/50 | loss: 0.0035 | CER: 0.1160 | Exact: 38.7%
Epoch  30/50 | loss: 0.0005 | CER: 0.1224 | Exact: 35.7%
Epoch  35/50 | loss: 0.0003 | CER: 0.1192 | Exact: 37.3%
Epoch  40/50 | loss: 0.0002 | CER: 0.1172 | Exact: 38.0%
Epoch  45/50 | loss: 0.0002 | CER: 0.1196 | Exact: 36.7%
Epoch  50/50 | loss: 0.0001 | CER: 0.1149 | Exact: 38.0%

Final CER: 0.1149 | Exact accuracy: 38.00%
Sample predictions:
  ✓ GT: A001BP54     | PRED: A001BP54
  ✓ GT: A001PC71     | PRED: A001PC71
  ✓ GT: A002KX152    | PRED: A002KX152
  ✗ GT: A002XC763    | PRED: A002XC35
  ✗ GT: A002XY89     | PRED: A002XM52
  ✗ GT: A003CX196    | PRED: K003CX35
  ✗ GT: A004OE23     | PRED: A004OE22
  ✗ GT: A005AX26     | PRED: A005

In [18]:
ckpt = {
    "state_dict": final_model.state_dict(),
    "vocab_size": vocab_size,
    "img_height": IMG_HEIGHT,
    "img_width":  IMG_WIDTH,
    "rnn_hidden": 256,
    "rnn_layers": 2,
    "cnn_channels": [64, 128, 256, 256],
    "final_cer":  final_cer,
    "final_exact": final_exact,
}
torch.save(ckpt, CHECKPOINT_DIR / "best_crnn.pt")

alphabet_data = {
    "alphabet": ALPHABET,
    "char2idx": char2idx,
    "idx2char": {str(k): v for k, v in idx2char.items()},
    "blank_idx": blank_idx,
    "vocab_size": vocab_size,
}
with open(CHECKPOINT_DIR / "alphabet.json", "w", encoding="utf-8") as f:
    json.dump(alphabet_data, f, ensure_ascii=False, indent=2)

print(f"Checkpoint saved: {CHECKPOINT_DIR / 'best_crnn.pt'}")
print(f"Alphabet saved:   {CHECKPOINT_DIR / 'alphabet.json'}")
print(f"Final CER: {final_cer:.4f} | Exact: {final_exact*100:.2f}%")

Checkpoint saved: C:\ITMO\designing_neural_network_architectures_2025_02\seminar_01\crnn_checkpoints\best_crnn.pt
Alphabet saved:   C:\ITMO\designing_neural_network_architectures_2025_02\seminar_01\crnn_checkpoints\alphabet.json
Final CER: 0.1149 | Exact: 38.00%


## Инференс — функция recognize()

In [ ]:
from torchvision.transforms.functional import to_pil_image

def load_crnn(checkpoint_path, alphabet_path, device=None):
    """Загружает CRNN из чекпоинта."""
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    with open(alphabet_path, "r", encoding="utf-8") as f:
        alph = json.load(f)

    ckpt = torch.load(checkpoint_path, map_location=device)
    m = CRNN(
        vocab_size=ckpt["vocab_size"],
        img_height=ckpt["img_height"],
        rnn_hidden=ckpt["rnn_hidden"],
        rnn_layers=ckpt["rnn_layers"],
        cnn_channels=ckpt["cnn_channels"],
    ).to(device)
    m.load_state_dict(ckpt["state_dict"])
    m.eval()

    idx2char_loaded = {int(k): v for k, v in alph["idx2char"].items()}
    blank = alph["blank_idx"]
    return m, idx2char_loaded, blank, ckpt["img_height"], ckpt["img_width"]


def recognize(img_input, model=None, idx2char_r=None, blank=0, h=32, w=128, dev=None):
    """
    Распознаёт номер на кропе.
    img_input: путь к файлу, PIL.Image или Tensor.
    """
    if dev is None:
        dev = device

    if model is None:
        model = final_model
        idx2char_r = idx2char
        blank = blank_idx

    if isinstance(img_input, (str, Path)):
        img = Image.open(img_input).convert("RGB")

    elif isinstance(img_input, torch.Tensor):
        # Tensor -> PIL
        if img_input.dim() == 4:
            img_input = img_input.squeeze(0)

        img = to_pil_image(img_input).convert("RGB")

    else:
        img = img_input.convert("RGB")

    tfm = T.Compose([
        T.Grayscale(1),
        T.Resize((h, w)),
        T.ToTensor(),
        T.Normalize((0.5,), (0.5,)),
    ])

    tensor = tfm(img).unsqueeze(0).to(dev)

    with torch.no_grad():
        logits = model(tensor)
        pred = logits.permute(1, 0, 2).argmax(2)[0]

    prev, out = None, []

    for s in pred.cpu().numpy().tolist():
        if s != blank and s != prev and s in idx2char_r:
            out.append(idx2char_r[s])
        prev = s

    return "".join(out)


# Тест инференса на 5 примерах из test set
print("Inference check:")
for img, lbl, txt in [test_dataset[i] for i in range(5)]:
    pred = recognize(img.unsqueeze(0).squeeze(0))
    ann_path = test_dataset.ann_paths[test_dataset.ann_paths.index(test_dataset.ann_paths[0])]

print("\nQuick test via DataLoader:")
final_model.eval()
for i in range(5):
    img_t, lbl, txt = test_dataset[i]
    with torch.no_grad():
        logits = final_model(img_t.unsqueeze(0).to(device))
        pred_seq = logits.permute(1, 0, 2).argmax(2)[0]
    pred_str = decode_sequence(pred_seq)
    mark = "✓" if pred_str == txt else "✗"
    print(f"  {mark} GT: {txt:12s} | PRED: {pred_str}")

Inference check:

Quick test via DataLoader:
  ✓ GT: A001BP54     | PRED: A001BP54
  ✓ GT: A001PC71     | PRED: A001PC71
  ✓ GT: A002KX152    | PRED: A002KX152
  ✗ GT: A002XC763    | PRED: A002XC35
  ✗ GT: A002XY89     | PRED: A002XM52
